# Docling vs MinerU: Q&A with Parsed Data

In [1]:
%load_ext autoreload
%autoreload 2

from IPython.display import display, Markdown
from pathlib import Path
import subprocess
import torch
from chonkie import NeuralChunker

In [2]:
pdf_path = "data/pdf/arxiv/2403.20331v2.pdf"
print("PDF exists:", Path(pdf_path).exists())
print("PDF path:", pdf_path)

PDF exists: True
PDF path: data/pdf/arxiv/2403.20331v2.pdf


## Parsed Text

* Same chunking strategy applied to both parsed text
* Same retrieval method
* Compare Q&A

#### Parse Text with Docling and MinerU

In [3]:
# Resolve the PDF path (relative to notebook dir if needed)
try:
    from IPython import get_ipython
    ip = get_ipython()
    nb_path = Path(ip.kernel.shell.user_ns.get("__vsc_ipynb_file__", ""))
    nb_dir = nb_path.parent if nb_path.exists() else Path.cwd()
except Exception:
    nb_dir = Path.cwd()

pdf_path = nb_dir / "data" / "pdf" / "arxiv" / "2403.20331v2.pdf"
print(f"PDF path: {pdf_path}")
print(f"PDF exists: {pdf_path.exists()}")

# ---------- Docling parse ----------
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True
pipeline_options.generate_table_images = False

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

docling_result = converter.convert(pdf_path)
docling_markdown = docling_result.document.export_to_markdown()
print(f"\nDocling markdown length: {len(docling_markdown):,} chars")

# ---------- MinerU parse ----------
mineru_output_dir = nb_dir / "data" / "pdf" / "arxiv" / "mineru_output"
mineru_output_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    "mineru",
    "-p", str(pdf_path),
    "-o", str(mineru_output_dir),
    "-b", "pipeline",
    "-l", "en",
    "-f", "true",
    "-t", "true",
]
print(f"\nRunning: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-1500:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-1500:])

parse_dir = mineru_output_dir / pdf_path.stem / "auto"
md_path = parse_dir / f"{pdf_path.stem}.md"

if md_path.exists():
    mineru_markdown = md_path.read_text(encoding="utf-8")
    print(f"\nMinerU markdown length: {len(mineru_markdown):,} chars")
else:
    mineru_markdown = ""
    print(f"\nMinerU markdown not found at {md_path}")

# Store for later cells
parsed_text = {
    "docling": docling_markdown,
    "mineru": mineru_markdown,
}

PDF path: /Users/hanhanwu/Documents/Github/Yokan/experiments_multimodal/pdf_parsing/data/pdf/arxiv/2403.20331v2.pdf
PDF exists: True


/Users/hanhanwu/Documents/Github/Yokan/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[INFO] 2026-07-20 22:54:22,001 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-20 22:54:22,007 [RapidOCR] download_file.py:60: File exists and is valid: /Users/hanhanwu/Documents/Github/Yokan/.venv/lib/python3.14/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-20 22:54:22,008 [RapidOCR] main.py:63: Using /Users/hanhanwu/Documents/Github/Yokan/.venv/lib/python3.14/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-20 22:54:22,020 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-20 22:54:22,021 [RapidOCR] download_file.py:60: File exists and is valid: /Users/hanhanwu/Documents/Github/Yokan/.venv/lib/python3.14/site-packages/rapidocr/mo


Docling markdown length: 187,228 chars

Running: mineru -p /Users/hanhanwu/Documents/Github/Yokan/experiments_multimodal/pdf_parsing/data/pdf/arxiv/2403.20331v2.pdf -o /Users/hanhanwu/Documents/Github/Yokan/experiments_multimodal/pdf_parsing/data/pdf/arxiv/mineru_output -b pipeline -l en -f true -t true
Start MinerU FastAPI Service: http://127.0.0.1:53555
API documentation: http://127.0.0.1:53555/docs


MinerU markdown length: 128,868 chars


In [4]:
# Use MPS on Apple Silicon (Mac M5)
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# Initialize the same NeuralChunker for both parsed texts
chunker = NeuralChunker(
    model="mirth/chonky_modernbert_base_1",
    device_map=device,
    min_characters_per_chunk=10,
)

# Apply the same chunking to both Docling and MinerU parsed text
docling_chunks = chunker.chunk(docling_markdown)
mineru_chunks = chunker.chunk(mineru_markdown)

print(f"Docling chunks: {len(docling_chunks)}")
print(f"MinerU chunks:  {len(mineru_chunks)}")

# Store chunk texts for later cells
chunked_text = {
    "docling": [c.text for c in docling_chunks],
    "mineru": [c.text for c in mineru_chunks],
}

Using device: mps


Device set to use mps
Token indices sequence length is longer than the specified maximum sequence length for this model (10569 > 8192). Running this sequence through the model will result in indexing errors


Docling chunks: 96
MinerU chunks:  55


## Retrieval & Answer

In [5]:
import json
from pathlib import Path

data_base = nb_dir / "data" / "pdf" / "arxiv"

queries = json.loads((data_base / "queries.json").read_text())
qrels = json.loads((data_base / "qrels.json").read_text())

paper_id = pdf_path.stem  # e.g. "2403.20331v2"

linked_query_ids = [
    qid for qid, rel in qrels.items()
    if rel["doc_id"] == paper_id
]

print(f"Queries linked to this paper: {len(linked_query_ids)}\n")
for qid in linked_query_ids:
    q = queries[qid]
    print(f"[{q['source']:20s}] [{q['type']:11s}]  {q['query'][:90]}")

text_query_lst = [queries[qid]["query"] for qid in linked_query_ids]
print(len(text_query_lst))


Queries linked to this paper: 10

[text                ] [extractive ]  What does MM-UPD Bench stand for?
[text                ] [extractive ]  Is detailed justification required when refining problems during the curation process?
[text                ] [extractive ]  Is there a strong correlation between OC-Dual accuracy and Dual accuracy?
[text                ] [extractive ]  Does self-reflection allow models to evaluate their own answers?
[text                ] [extractive ]  What activity involves using ropes and harnesses on a cliff face?
[text                ] [abstractive]  Why is Dual accuracy considered effective for assessing LMMs' reliability?
[text                ] [abstractive]  What is the purpose of Unsolvable Problem Detection (UPD)?
[text                ] [extractive ]  Does UPD-specific training affect the performance of general tasks?
[text                ] [extractive ]  Do closed-source models offer a better response than open-source models in this evaluation
[text

In [6]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI
from rank_bm25 import BM25Okapi
import tiktoken

load_dotenv("../../.env")  # adjust if your .env is elsewhere
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

EMBEDDING_MODEL = "text-embedding-3-small"
MAX_TOKENS = 8191  # OpenAI limit for text-embedding-3-small

_tokenizer = tiktoken.get_encoding("cl100k_base")


def embed_texts(texts: list[str]) -> np.ndarray:
    """Embed a list of texts using OpenAI text-embedding-3-small."""
    response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return np.array([item.embedding for item in response.data], dtype=float)


def cosine_sim_batch(query_vecs: np.ndarray, chunk_vecs: np.ndarray) -> np.ndarray:
    """Vectorized cosine similarity: (Q, D) x (N, D) -> (Q, N)."""
    query_norms = np.linalg.norm(query_vecs, axis=1, keepdims=True)
    chunk_norms = np.linalg.norm(chunk_vecs, axis=1, keepdims=True)
    dot = query_vecs @ chunk_vecs.T
    return dot / np.maximum(query_norms * chunk_norms.T, 1e-10)


def rrf_fuse(rankings: list[list[int]], k: int = 60) -> dict[int, float]:
    """Reciprocal Rank Fusion across multiple ranked lists."""
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, idx in enumerate(ranking):
            scores[idx] += 1.0 / (k + rank + 1)
    return scores


def truncate_to_token_limit(text: str, max_tokens: int = MAX_TOKENS) -> str:
    """Truncate text to stay within OpenAI's token limit using tiktoken."""
    tokens = _tokenizer.encode(text)
    if len(tokens) <= max_tokens:
        return text
    return _tokenizer.decode(tokens[:max_tokens])


def hybrid_top1(chunks: list[str], queries: list[str]) -> list[tuple[str, float]]:
    """Return the top-1 chunk for each query using semantic + BM25 hybrid search."""
    # Build BM25 index on full chunks
    tokenized_corpus = [text.lower().split() for text in chunks]
    bm25 = BM25Okapi(tokenized_corpus)

    # Embed queries in batch
    query_embeddings = embed_texts(queries)

    # Embed chunks in batch after truncation
    truncated_chunks = [truncate_to_token_limit(chunk) for chunk in chunks]
    chunk_embeddings = embed_texts(truncated_chunks)

    # All cosine similarities at once: (Q, N)
    sem_scores_all = cosine_sim_batch(query_embeddings, chunk_embeddings)

    top1_results = []
    for q_idx, query in enumerate(queries):
        sem_scores = sem_scores_all[q_idx]
        bm25_scores = bm25.get_scores(query.lower().split())

        sem_ranking = np.argsort(-sem_scores).tolist()
        kw_ranking = np.argsort(-bm25_scores).tolist()

        fused = rrf_fuse([sem_ranking, kw_ranking])
        top_idx = max(fused, key=lambda i: fused[i])

        top1_results.append((chunks[top_idx], fused[top_idx]))

    return top1_results


def truncate_words(text, n=300):
    if isinstance(text, str):
        words = text.split()
        return " ".join(words[:n]) + ("..." if len(words) > n else "")
    return text


# Run retrieval on both Docling and MinerU chunks
docling_top1 = hybrid_top1(chunked_text["docling"], text_query_lst)
mineru_top1 = hybrid_top1(chunked_text["mineru"], text_query_lst)

# Build comparison table
comparison_rows = []
for q, (d_chunk, d_score), (m_chunk, m_score) in zip(text_query_lst, docling_top1, mineru_top1):
    comparison_rows.append({
        "query": q,
        "top1_retrieved_chunk_docling": d_chunk,
        "top1_retrieved_chunk_mineru": m_chunk,
    })

df = pd.DataFrame(comparison_rows)
display_df = df.copy()
for col in ["top1_retrieved_chunk_docling", "top1_retrieved_chunk_mineru"]:
    display_df[col] = display_df[col].apply(lambda x: truncate_words(x, 30))

pd.set_option("display.max_colwidth", None)
display(display_df)

,query,top1_retrieved_chunk_docling,top1_retrieved_chunk_mineru
0,What does MM-UPD Bench stand for?,"## Unsolvable Problem Detection: Robust Understanding Evaluation for Large Multimodal Models Atsuyuki Miyai 1 Jingkang Yang 2 Jingyang Zhang 3 Yifei Ming 4 Qing Yu 1 , 5 Go Irie...",<table><tr><td rowspan=1 colspan=1>#8</td><td rowspan=1 colspan=1>#9</td><td rowspan=1 colspan=2>#10</td><td rowspan=1 colspan=2>#11</td><td rowspan=1 colspan=2>#12</td><td rowspan=1 colspan=1>#13</td></tr><tr><td rowspan=1 colspan=1>PhysicalRelation</td><td rowspan=1 colspan=1>SocialRelation</td><td rowspan=1 colspan=2>IdentityReasoning</td><td rowspan=1 colspan=2>FunctionReasoning</td><td rowspan=1 colspan=2>PhysicalPropertyReasoning</td><td rowspan=1 colspan=1>StructuralizedImage-textUnderstanding</td></tr><tr><td rowspan=3 colspan=9>#14 #15 #16 #17...
1,Is detailed justification required when refining problems during the curation process?,"Result. Weshow the comparison results in Table D. Based on these results, in contrast to MM-UPD, we could not verify the efficacy of either the Option or Instruction approaches. This...","Clarification of the Semantics of the Options. We clarify the meaning of the options. Specifically, some questions in #6: Attribute Comparison have “Can’t judge”. “Can’t judge” means that “I can’t..."
2,Is there a strong correlation between OC-Dual accuracy and Dual accuracy?,Figure A: Relationship between OC-Dual accuracy and Dual accuracy. <!-- image -->,"Table 1: Comparison results of the overall Dual accuracy for the base setting, additional-option setting, and additional-instruction setting. The “Orig” (Original Standard) value is the upper bound of Dual accuracy...."
3,Does self-reflection allow models to evaluate their own answers?,"Result. Weshow the comparison results in Table D. Based on these results, in contrast to MM-UPD, we could not verify the efficacy of either the Option or Instruction approaches. This...",ail ## C.1 Experimental Setup Computing Infrastructures. We conduct all our evaluations of open-source models on a single NVIDIA A100 (80GB) GPU. HyperParameters of LMM Inference. We set a temperature...
4,What activity involves using ropes and harnesses on a cliff face?,Q. What's the function of the demonstrated object? ## A. running - B. Play football - C. Play basketball ## GPT-4o None of the provided options are correct . The...,"Marah Abdin, Sam Ade Jacobs, Ammar Ahmad Awan, Jyoti Aneja, Ahmed Awadallah, Hany Awadalla, Nguyen Bach, Amit Bahree, Arash Bakhtiari, Harkirat Behl, et al. 2024. Phi-3 technical report: A highly..."
5,Why is Dual accuracy considered effective for assessing LMMs' reliability?,"F3: UPD Score is Significantly Lower than Standard in Base and Solution Varies by LMMs. Fig. 3 shows the Standard (blue) and UPD (red) accuracy. The performance was compared, with...","Table 1: Comparison results of the overall Dual accuracy for the base setting, additional-option setting, and additional-instruction setting. The “Orig” (Original Standard) value is the upper bound of Dual accuracy...."
6,What is the purpose of Unsolvable Problem Detection (UPD)?,"## Unsolvable Problem Detection: Robust Understanding Evaluation for Large Multimodal Models Atsuyuki Miyai 1 Jingkang Yang 2 Jingyang Zhang 3 Yifei Ming 4 Qing Yu 1 , 5 Go Irie...","# Unsolvable Problem Detection: Robust Understanding Evaluation for Large Multimodal Models Atsuyuki Miyai<sup>1</sup> Jingkang Yang<sup>2</sup> Jingyang Zhang<sup>3</sup> Yifei Ming<sup>4</sup> Qing Yu<sup>1,5</sup> Go Irie<sup>6</sup> Yixuan Li<sup>4</sup> Hai Li<sup>3</sup> Ziwei Liu<sup>2</sup> Kiyoharu..."
7,Does UPD-specific training affect the performance of general tasks?,"## D.1 Setup Dataset. For the dataset, we use a subset of an open-knowledge VQA dataset, AOKVQA (Schwenk et al., 2022). It is a singlechoice type VQA dataset that has...","Table D: Performance comparison on MMMU-AAD. We report overall Dual accuracy. The values in () represent Standard accura

In [7]:

from openai import OpenAI

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)
GROQ_MODEL = "openai/gpt-oss-20b"


def generate_answer(client, model, query: str, context: str) -> str:
    """Generate an answer for a query given a retrieved context chunk."""
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": """You are a helpful assistant. 
                Answer the question based solely on the provided context. Be concise.
                If can't find the answer in the context, say "Not found in context."
                """,
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {query}",
            },
        ],
    )
    return response.choices[0].message.content.strip()


df["answer_docling"] = [
    generate_answer(groq_client, GROQ_MODEL, row["query"], row["top1_retrieved_chunk_docling"])
    for _, row in df.iterrows()
]
df["answer_mineru"] = [
    generate_answer(groq_client, GROQ_MODEL, row["query"], row["top1_retrieved_chunk_mineru"])
    for _, row in df.iterrows()
]

pd.set_option("display.max_colwidth", 120)
display(df)


,query,top1_retrieved_chunk_docling,top1_retrieved_chunk_mineru,answer_docling,answer_mineru
0,What does MM-UPD Bench stand for?,## Unsolvable Problem Detection: Robust Understanding Evaluation for Large Multimodal Models\n\nAtsuyuki Miyai 1 Jin...,\n\n<table><tr><td rowspan=1 colspan=1>#8</td><td rowspan=1 colspan=1>#9</td><td rowspan=1 colspan=2>#10</td><td row...,**MM‑UPD Bench** stands for the *Multimodal Unsolvable Problem Detection Benchmark*.,Not found in context.
1,Is detailed justification required when refining problems during the curation process?,"\nResult. Weshow the comparison results in Table D. Based on these results, in contrast to MM-UPD, we could not veri...","\n\nClarification of the Semantics of the Options. We clarify the meaning of the options. Specifically, some questio...",Not found in context.,"Yes – annotators must record a detailed justification (e.g., a comment explaining why an image relates to a question..."
2,Is there a strong correlation between OC-Dual accuracy and Dual accuracy?,Figure A: Relationship between OC-Dual accuracy and Dual accuracy.\n\n<!-- image -->,"\n\nTable 1: Comparison results of the overall Dual accuracy for the base setting, additional-option setting, and ad...",Not found in context.,Not found in context.
3,Does self-reflection allow models to evaluate their own answers?,"\nResult. Weshow the comparison results in Table D. Based on these results, in contrast to MM-UPD, we could not veri...",ail\n\n## C.1 Experimental Setup\n\nComputing Infrastructures. We conduct all our evaluations of open-source models ...,"Yes. The experiments show that self‑reflection lets the model judge and evaluate its own answer, and it can withdraw...",Yes – the paper notes that self‑reflection “has been shown that LLMs might have preliminary capabilities for judging...
4,What activity involves using ropes and harnesses on a cliff face?,\n\nQ. What's the function of the demonstrated object?\n\n## A. running\n\n- B. Play football\n- C. Play basketball\...,"\n\nMarah Abdin, Sam Ade Jacobs, Ammar Ahmad Awan, Jyoti Aneja, Ahmed Awadallah, Hany Awadalla, Nguyen Bach, Amit Ba...",Rock climbing.,Not found in context.
5,Why is Dual accuracy considered effective for assessing LMMs' reliability?,\nF3: UPD Score is Significantly Lower than Standard in Base and Solution Varies by LMMs. Fig. 3 shows the Standard ...,"\n\nTable 1: Comparison results of the overall Dual accuracy for the base setting, additional-option setting, and ad...",Not found in context.,Dual accuracy is effective because it directly measures how far a model’s performance is from the best‑possible (Upp...
6,What is the purpose of Unsolvable Problem Detection (UPD)?,## Unsolvable Problem Detection: Robust Understanding Evaluation for Large Multimodal Models\n\nAtsuyuki Miyai 1 Jin...,# Unsolvable Problem Detection: Robust Understanding Evaluation for Large Multimodal Models\n\nAtsuyuki Miyai<sup>1<...,Unsolvable Problem Detection (UPD) is designed to evaluate a Large Multimodal Model’s robust understanding by testin...,Unsolvable Problem Detection (UPD) is a task that tests whether a Large Multimodal Model can recognize when a multip...
7,Does UPD-specific training affect the performance of general tasks?,"\n\n## D.1 Setup\n\nDataset. For the dataset, we use a subset of an open-knowledge VQA dataset, AOKVQA (Schwenk et a...",\n\nTable D: Performance comparison on MMMU-AAD. We report overall Dual accuracy. The values in () represent Standar...,"Yes. The results show that while UPD‑specific instruction tuning boosts UPD performance, it can hurt performance on ...","Yes. The text notes that while UPD‑specific instruction tuning improves UPD accuracy, it can **degrade performance o..."
8,Do closed-source models offer a better response than open-source models in this evaluation?,\nF1: Different Performance Trends of MMBench and MM-UPD Bench. Table 1 shows that the performance trends of MMBench...,\nF1: Different Performance Trends of MM-Bench and

## Evaluation

In [8]:
%load_ext autoreload
%autoreload 2

import sys
import importlib
import utils as eval_utils

from langchain_groq import ChatGroq

if "eval_llm" not in dir():
    eval_llm = ChatGroq(
        api_key=os.getenv("GROQ_API_KEY"),
        model=GROQ_MODEL,
        temperature=0,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
# Evaluate retrieval relevancy for docling and mineru retrieved chunks
docling_eval_df = await eval_utils.get_retrieval_relevancy_output_async(
    df[['query', 'top1_retrieved_chunk_docling']].rename(columns={'top1_retrieved_chunk_docling': 'retrieved_chunk'}),
    eval_llm,
    chunk_col='retrieved_chunk',
    score_col='retrieval_score_docling',
    reasoning_col='rr_reasoning_docling',
)

mineru_eval_df = await eval_utils.get_retrieval_relevancy_output_async(
    df[['query', 'top1_retrieved_chunk_mineru']].rename(columns={'top1_retrieved_chunk_mineru': 'retrieved_chunk'}),
    eval_llm,
    chunk_col='retrieved_chunk',
    score_col='retrieval_score_mineru',
    reasoning_col='rr_reasoning_mineru',
)

# Merge scores back
eval_df = df[['query', 'top1_retrieved_chunk_docling', 'top1_retrieved_chunk_mineru']].copy()
eval_df['retrieval_score_docling'] = docling_eval_df['retrieval_score_docling'].values
eval_df['rr_reasoning_docling'] = docling_eval_df['rr_reasoning_docling'].values
eval_df['retrieval_score_mineru'] = mineru_eval_df['retrieval_score_mineru'].values
eval_df['rr_reasoning_mineru'] = mineru_eval_df['rr_reasoning_mineru'].values

pd.set_option("display.max_colwidth", 120)
display(eval_df[['query', 'top1_retrieved_chunk_docling', 'retrieval_score_docling',
    'rr_reasoning_docling',
    'top1_retrieved_chunk_mineru', 'retrieval_score_mineru', 'rr_reasoning_mineru']])

,query,top1_retrieved_chunk_docling,retrieval_score_docling,rr_reasoning_docling,top1_retrieved_chunk_mineru,retrieval_score_mineru,rr_reasoning_mineru
0,What does MM-UPD Bench stand for?,## Unsolvable Problem Detection: Robust Understanding Evaluation for Large Multimodal Models\n\nAtsuyuki Miyai 1 Jin...,1,"The chunk mentions MM‑UPD Bench as a benchmark for unsolvable problem detection, but it does not explain what the ac...",\n\n<table><tr><td rowspan=1 colspan=1>#8</td><td rowspan=1 colspan=1>#9</td><td rowspan=1 colspan=2>#10</td><td row...,1,The chunk references MM‑UPD Bench but only shows a mapping table; it does not explain what MM‑UPD Bench stands for.
1,Is detailed justification required when refining problems during the curation process?,"\nResult. Weshow the comparison results in Table D. Based on these results, in contrast to MM-UPD, we could not veri...",0,"The chunk discusses LMM evaluation methods and experimental details, with no mention of justification or problem ref...","\n\nClarification of the Semantics of the Options. We clarify the meaning of the options. Specifically, some questio...",3,"The chunk explicitly describes that when refining problems, annotators record detailed justifications in comments, d..."
2,Is there a strong correlation between OC-Dual accuracy and Dual accuracy?,Figure A: Relationship between OC-Dual accuracy and Dual accuracy.\n\n<!-- image -->,2,"The chunk references a figure titled ""Relationship between OC‑Dual accuracy and Dual accuracy,"" indicating relevance...","\n\nTable 1: Comparison results of the overall Dual accuracy for the base setting, additional-option setting, and ad...",1,"The chunk discusses Dual accuracy but does not mention OC‑Dual accuracy or any correlation between the two metrics, ..."
3,Does self-reflection allow models to evaluate their own answers?,"\nResult. Weshow the comparison results in Table D. Based on these results, in contrast to MM-UPD, we could not veri...",3,"The chunk explicitly discusses self‑reflection as a method for models to evaluate their own responses, citing studie...",ail\n\n## C.1 Experimental Setup\n\nComputing Infrastructures. We conduct all our evaluations of open-source models ...,2,"The chunk discusses self‑reflection as a method for models to evaluate their own responses, indicating preliminary c..."
4,What activity involves using ropes and harnesses on a cliff face?,\n\nQ. What's the function of the demonstrated object?\n\n## A. running\n\n- B. Play football\n- C. Play basketball\...,3,"The chunk explicitly mentions ""rock climbing"" as an activity that uses ropes and harnesses on a cliff face, directly...","\n\nMarah Abdin, Sam Ade Jacobs, Ammar Ahmad Awan, Jyoti Aneja, Ahmed Awadallah, Hany Awadalla, Nguyen Bach, Amit Ba...",0,"The chunk contains a bibliography of AI research papers and does not mention ropes, harnesses, cliff faces, or any c..."
5,Why is Dual accuracy considered effective for assessing LMMs' reliability?,\nF3: UPD Score is Significantly Lower than Standard in Base and Solution Varies by LMMs. Fig. 3 shows the Standard ...,2,"The chunk mentions Dual accuracy and compares UPD vs Standard accuracy, indicating its use in evaluating LMMs, but i...","\n\nTable 1: Comparison results of the overall Dual accuracy for the base setting, additional-option setting, and ad...",1,"The chunk mentions Dual accuracy and presents comparison results, but it does not explain why Dual accuracy is consi..."
6,What is the purpose of Unsolvable Problem Detection (UPD)?,## Unsolvable Problem Detection: Robust Understanding Evaluation for Large Multimodal Models\n\nAtsuyuki Miyai 1 Jin...,3,"The chunk explains UPD’s goal of testing LMMs’ ability to refuse unsolvable MCQA problems, detailing its purpose and...",# Unsolvable Problem Detection: Robust Understanding Evaluation for Large Multimodal Models\n\nAtsuyuki Miyai<sup>1<...,3,The chunk explicitly defines UPD as a task to evaluate LMMs’ ability to recognize and withhold answers fo

In [10]:
# Merge answers into eval_df
eval_df['answer_docling'] = df['answer_docling'].values
eval_df['answer_mineru'] = df['answer_mineru'].values

# Evaluate answer quality for docling
docling_aq_df = await eval_utils.get_answer_quality_output_async(
    eval_df[['query', 'top1_retrieved_chunk_docling', 'retrieval_score_docling', 'answer_docling']].rename(columns={
        'top1_retrieved_chunk_docling': 'retrieved_chunk',
        'retrieval_score_docling': 'retrieval_relevancy_score',
        'answer_docling': 'answer',
    }),
    eval_llm,
    score_col='answer_quality_score_docling',
    grounded_col='aq_grounded_docling',
    reasoning_col='aq_reasoning_docling',
)

# Evaluate answer quality for mineru
mineru_aq_df = await eval_utils.get_answer_quality_output_async(
    eval_df[['query', 'top1_retrieved_chunk_mineru', 'retrieval_score_mineru', 'answer_mineru']].rename(columns={
        'top1_retrieved_chunk_mineru': 'retrieved_chunk',
        'retrieval_score_mineru': 'retrieval_relevancy_score',
        'answer_mineru': 'answer',
    }),
    eval_llm,
    score_col='answer_quality_score_mineru',
    grounded_col='aq_grounded_mineru',
    reasoning_col='aq_reasoning_mineru',
)

# Merge answer quality scores back into eval_df
eval_df['answer_quality_score_docling'] = docling_aq_df['answer_quality_score_docling'].values
eval_df['aq_grounded_docling'] = docling_aq_df['aq_grounded_docling'].values
eval_df['aq_reasoning_docling'] = docling_aq_df['aq_reasoning_docling'].values
eval_df['answer_quality_score_mineru'] = mineru_aq_df['answer_quality_score_mineru'].values
eval_df['aq_grounded_mineru'] = mineru_aq_df['aq_grounded_mineru'].values
eval_df['aq_reasoning_mineru'] = mineru_aq_df['aq_reasoning_mineru'].values

# Display all retrieval and answer eval results
pd.set_option("display.max_colwidth", 120)
eval_df = eval_df[[
    'query',
    'top1_retrieved_chunk_docling', 'retrieval_score_docling', 'rr_reasoning_docling',
    'answer_docling', 'answer_quality_score_docling', 'aq_grounded_docling', 'aq_reasoning_docling',
    'top1_retrieved_chunk_mineru', 'retrieval_score_mineru', 'rr_reasoning_mineru',
    'answer_mineru', 'answer_quality_score_mineru', 'aq_grounded_mineru', 'aq_reasoning_mineru',
]]
display(eval_df)


,query,top1_retrieved_chunk_docling,retrieval_score_docling,rr_reasoning_docling,answer_docling,answer_quality_score_docling,aq_grounded_docling,aq_reasoning_docling,top1_retrieved_chunk_mineru,retrieval_score_mineru,rr_reasoning_mineru,answer_mineru,answer_quality_score_mineru,aq_grounded_mineru,aq_reasoning_mineru
0,What does MM-UPD Bench stand for?,## Unsolvable Problem Detection: Robust Understanding Evaluation for Large Multimodal Models\n\nAtsuyuki Miyai 1 Jin...,1,"The chunk mentions MM‑UPD Bench as a benchmark for unsolvable problem detection, but it does not explain what the ac...",**MM‑UPD Bench** stands for the *Multimodal Unsolvable Problem Detection Benchmark*.,3,None,"The answer correctly expands MM‑UPD Bench as ""Multimodal Unsolvable Problem Detection Benchmark,"" fully addressing t...",\n\n<table><tr><td rowspan=1 colspan=1>#8</td><td rowspan=1 colspan=1>#9</td><td rowspan=1 colspan=2>#10</td><td row...,1,The chunk references MM‑UPD Bench but only shows a mapping table; it does not explain what MM‑UPD Bench stands for.,Not found in context.,0,None,The answer does not provide the expansion of MM‑UPD Bench and therefore fails to address the user’s question.
1,Is detailed justification required when refining problems during the curation process?,"\nResult. Weshow the comparison results in Table D. Based on these results, in contrast to MM-UPD, we could not veri...",0,"The chunk discusses LMM evaluation methods and experimental details, with no mention of justification or problem ref...",Not found in context.,0,None,The answer does not address the user’s question about detailed justification during problem refinement.,"\n\nClarification of the Semantics of the Options. We clarify the meaning of the options. Specifically, some questio...",3,"The chunk explicitly describes that when refining problems, annotators record detailed justifications in comments, d...","Yes – annotators must record a detailed justification (e.g., a comment explaining why an image relates to a question...",3,True,The answer correctly states that detailed justification is required for refining problems and describes the consensu...
2,Is there a strong correlation between OC-Dual accuracy and Dual accuracy?,Figure A: Relationship between OC-Dual accuracy and Dual accuracy.\n\n<!-- image -->,2,"The chunk references a figure titled ""Relationship between OC‑Dual accuracy and Dual accuracy,"" indicating relevance...",Not found in context.,0,False,"The answer does not address the question about the correlation between OC‑Dual accuracy and Dual accuracy, so it sco...","\n\nTable 1: Comparison results of the overall Dual accuracy for the base setting, additional-option setting, and ad...",1,"The chunk discusses Dual accuracy but does not mention OC‑Dual accuracy or any correlation between the two metrics, ...",Not found in context.,0,None,The answer does not address the user’s question about the correlation between OC-Dual accuracy and Dual accuracy.
3,Does self-reflection allow models to evaluate their own answers?,"\nResult. Weshow the comparison results in Table D. Based on these results, in contrast to MM-UPD, we could not veri...",3,"The chunk explicitly discusses self‑reflection as a method for models to evaluate their own responses, citing studie...","Yes. The experiments show that self‑reflection lets the model judge and evaluate its own answer, and it can withdraw...",3,True,"The answer directly confirms that self‑reflection enables models to judge and evaluate their own answers, matching t...",ail\n\n## C.1 Experimental Setup\n\nComputing Infrastructures. We conduct all our evaluations of open-source models ...,2,"The chunk discusses self‑reflection as a method for models to evaluate their own responses, indicating preliminary c...",Yes – the paper notes that self‑reflection “has been shown that LLMs might have preliminary capabilities for judging...,3,True,"The answer directly confirms that self‑reflection enables models to evaluate their

In [11]:
eval_df.to_excel("docling_vs_mineru_qa_comparison.xlsx", index=False)